# Interactive Network Visualization with topologic_fast

This notebook demonstrates interactive network visualization techniques using topologic_fast.

We will:
1. Create a complex CellComplex (office building floorplan)
2. Extract the dual graph representing room connectivity
3. Visualize interactively with Plotly
4. Group and color nodes by room type

**Note:** This is adapted from topologicpy's Pyvis tutorial. The original uses the Pyvis library for interactive HTML network graphs. Here we use Plotly for similar interactive visualizations that work directly in Jupyter notebooks.

In [ ]:
# Import libraries
import topologic_fast as tf
import plotly.graph_objects as go
import numpy as np

## Create a CellComplex Building

We'll create a multi-room building similar to the original Pyvis example, with rooms of different types.

In [ ]:
# Building parameters
floor_height = 3.0

# Room definitions with types and colors
rooms = []
room_names = []
room_types = []
room_colors = []

# Color scheme by room type (thermal-like colorscale)
color_map = {
    'exterior_apertures': '#1a1a2e',   # Dark blue
    'exterior_surfaces': '#16213e',    # Navy
    'internal_apertures': '#0f3460',   # Blue
    'internal_surfaces': '#e94560',    # Red-pink
    'cells': '#ffc93c'                 # Gold/Yellow
}

def add_room(x, y, w, l, name, rtype):
    """Helper to add a room"""
    room = tf.Cell.Box(x, y, 0, w, l, floor_height)
    rooms.append(room)
    room_names.append(name)
    room_types.append(rtype)
    room_colors.append(color_map.get(rtype, '#808080'))
    return room

# Create a building layout (5x3 + some merged cells)
# Main building shell with internal divisions
add_room(0, 0, 2, 2, 'Room A1', 'cells')
add_room(2, 0, 2, 2, 'Room A2', 'cells')
add_room(4, 0, 2, 2, 'Room A3', 'cells')
add_room(6, 0, 2, 2, 'Room A4', 'cells')
add_room(8, 0, 2, 2, 'Room A5', 'cells')

add_room(0, 2, 4, 2, 'Corridor North', 'internal_surfaces')
add_room(4, 2, 6, 2, 'Corridor South', 'internal_surfaces')

add_room(0, 4, 2, 2, 'Room B1', 'cells')
add_room(2, 4, 2, 2, 'Room B2', 'cells')
add_room(4, 4, 2, 2, 'Room B3', 'cells')
add_room(6, 4, 2, 2, 'Room B4', 'cells')
add_room(8, 4, 2, 2, 'Room B5', 'cells')

# Additional floor
add_room(0, 0, 2, 2, 'Room C1', 'cells')
add_room(2, 0, 2, 2, 'Room C2', 'cells')
add_room(4, 0, 2, 2, 'Room C3', 'cells')
add_room(6, 0, 2, 2, 'Room C4', 'cells')
add_room(8, 0, 2, 2, 'Room C5', 'cells')

# Move upper floor cells to z=3
for i in range(12, 17):
    rooms[i] = tf.Cell.Box(
        (i-12)*2, 0, floor_height,
        2, 2, floor_height
    )

print(f"Created {len(rooms)} rooms")

# Create CellComplex
building = tf.CellComplex.ByCells(rooms)

print(f"\nCellComplex Statistics:")
print(f"  Cells: {building.NumCells()}")
print(f"  Total Volume: {building.Volume():.1f} m^3")

## Create the Dual Graph

The graph represents room connectivity where shared walls become edges.

In [ ]:
# Create dual graph
graph = tf.Graph.ByTopology(building)

print(f"Graph Statistics:")
print(f"  Vertices: {graph.Order()}")
print(f"  Edges: {graph.Size()}")
print(f"  Density: {graph.Density():.3f}")

## Interactive 2D Network Visualization

Create an interactive network diagram similar to Pyvis output.

In [ ]:
def create_network_visualization(graph, node_labels, node_types, type_colors, title="Network Graph"):
    """
    Create an interactive network visualization with Plotly.
    Nodes are colored by their group/type.
    """
    vertices = graph.Vertices()
    edges = graph.Edges()
    
    # Get coordinates
    coords = [v.Coordinates() for v in vertices]
    degrees = [graph.VertexDegree(v) for v in vertices]
    
    fig = go.Figure()
    
    # Draw edges with hover info
    for edge in edges:
        edge_verts = edge.Vertices()
        if len(edge_verts) == 2:
            p1 = edge_verts[0].Coordinates()
            p2 = edge_verts[1].Coordinates()
            
            # Find node indices for edge labels
            idx1 = idx2 = -1
            for i, v in enumerate(vertices):
                vc = v.Coordinates()
                if abs(p1[0]-vc[0]) < 0.01 and abs(p1[1]-vc[1]) < 0.01:
                    idx1 = i
                if abs(p2[0]-vc[0]) < 0.01 and abs(p2[1]-vc[1]) < 0.01:
                    idx2 = i
            
            edge_label = ""
            if idx1 >= 0 and idx2 >= 0 and idx1 < len(node_labels) and idx2 < len(node_labels):
                edge_label = f"{node_labels[idx1]} <-> {node_labels[idx2]}"
            
            fig.add_trace(go.Scatter(
                x=[p1[0], p2[0]],
                y=[p1[1], p2[1]],
                mode='lines',
                line=dict(color='rgba(100,100,100,0.4)', width=2),
                hovertext=edge_label,
                hoverinfo='text' if edge_label else 'skip',
                showlegend=False
            ))
    
    # Group nodes by type for legend
    unique_types = list(set(node_types))
    
    for ntype in unique_types:
        indices = [i for i, t in enumerate(node_types) if t == ntype]
        if not indices:
            continue
            
        x = [coords[i][0] for i in indices]
        y = [coords[i][1] for i in indices]
        sizes = [15 + degrees[i] * 5 for i in indices]
        labels = [node_labels[i] if i < len(node_labels) else f"Node {i}" for i in indices]
        degs = [degrees[i] for i in indices]
        
        color = type_colors.get(ntype, '#808080')
        
        fig.add_trace(go.Scatter(
            x=x, y=y,
            mode='markers',
            marker=dict(
                size=sizes,
                color=color,
                line=dict(color='white', width=2)
            ),
            name=ntype,
            hovertext=[f"{labels[j]}<br>Type: {ntype}<br>Degree: {degs[j]}" for j in range(len(indices))],
            hoverinfo='text'
        ))
    
    fig.update_layout(
        title=dict(text=title, font=dict(size=20)),
        xaxis=dict(
            showgrid=False, 
            zeroline=False, 
            showticklabels=False,
            title=''
        ),
        yaxis=dict(
            showgrid=False, 
            zeroline=False, 
            showticklabels=False,
            scaleanchor='x',
            title=''
        ),
        plot_bgcolor='white',
        width=900,
        height=600,
        legend=dict(
            title='Room Type',
            x=1.02,
            y=1,
            bgcolor='rgba(255,255,255,0.8)'
        ),
        hovermode='closest'
    )
    
    return fig

# Create visualization
fig_network = create_network_visualization(
    graph, 
    room_names, 
    room_types, 
    color_map,
    "Building Room Connectivity Network"
)
fig_network.show()

## 3D Network Visualization

Since our building has multiple floors, let's visualize the network in 3D.

In [ ]:
def create_3d_network(graph, node_labels, node_types, type_colors):
    """
    Create a 3D network visualization.
    """
    vertices = graph.Vertices()
    edges = graph.Edges()
    
    coords = [v.Coordinates() for v in vertices]
    degrees = [graph.VertexDegree(v) for v in vertices]
    
    fig = go.Figure()
    
    # Draw edges
    for edge in edges:
        edge_verts = edge.Vertices()
        if len(edge_verts) == 2:
            p1 = edge_verts[0].Coordinates()
            p2 = edge_verts[1].Coordinates()
            fig.add_trace(go.Scatter3d(
                x=[p1[0], p2[0]],
                y=[p1[1], p2[1]],
                z=[p1[2], p2[2]],
                mode='lines',
                line=dict(color='rgba(150,150,150,0.5)', width=3),
                showlegend=False,
                hoverinfo='skip'
            ))
    
    # Group nodes by type
    unique_types = list(set(node_types))
    
    for ntype in unique_types:
        indices = [i for i, t in enumerate(node_types) if t == ntype]
        if not indices:
            continue
            
        x = [coords[i][0] for i in indices]
        y = [coords[i][1] for i in indices]
        z = [coords[i][2] for i in indices]
        sizes = [8 + degrees[i] * 2 for i in indices]
        labels = [node_labels[i] if i < len(node_labels) else f"Node {i}" for i in indices]
        degs = [degrees[i] for i in indices]
        
        color = type_colors.get(ntype, '#808080')
        
        fig.add_trace(go.Scatter3d(
            x=x, y=y, z=z,
            mode='markers',
            marker=dict(
                size=sizes,
                color=color,
                line=dict(color='white', width=1)
            ),
            name=ntype,
            hovertext=[f"{labels[j]}<br>Type: {ntype}<br>Degree: {degs[j]}" for j in range(len(indices))],
            hoverinfo='text'
        ))
    
    fig.update_layout(
        title='3D Room Connectivity Network',
        scene=dict(
            xaxis=dict(title='X', showgrid=True),
            yaxis=dict(title='Y', showgrid=True),
            zaxis=dict(title='Z (Floor)', showgrid=True),
            aspectmode='data',
            camera=dict(eye=dict(x=1.5, y=-1.5, z=1.0))
        ),
        width=900,
        height=700,
        legend=dict(title='Room Type', x=1.02, y=1)
    )
    
    return fig

fig_3d = create_3d_network(graph, room_names, room_types, color_map)
fig_3d.show()

## Node Size by Degree (Connectivity)

Create a visualization where node size represents how connected each room is.

In [ ]:
def visualize_by_degree(graph, labels):
    """
    Visualize network with nodes sized and colored by degree.
    """
    vertices = graph.Vertices()
    edges = graph.Edges()
    
    coords = [v.Coordinates() for v in vertices]
    degrees = [graph.VertexDegree(v) for v in vertices]
    max_degree = max(degrees) if degrees else 1
    
    fig = go.Figure()
    
    # Draw edges
    for edge in edges:
        edge_verts = edge.Vertices()
        if len(edge_verts) == 2:
            p1 = edge_verts[0].Coordinates()
            p2 = edge_verts[1].Coordinates()
            fig.add_trace(go.Scatter(
                x=[p1[0], p2[0]],
                y=[p1[1], p2[1]],
                mode='lines',
                line=dict(color='rgba(200,200,200,0.5)', width=1),
                showlegend=False,
                hoverinfo='skip'
            ))
    
    # Draw nodes
    x = [c[0] for c in coords]
    y = [c[1] for c in coords]
    sizes = [20 + (d / max_degree) * 30 for d in degrees]
    
    fig.add_trace(go.Scatter(
        x=x, y=y,
        mode='markers+text',
        marker=dict(
            size=sizes,
            color=degrees,
            colorscale='Thermal',
            colorbar=dict(title='Degree'),
            line=dict(color='white', width=2)
        ),
        text=[str(d) for d in degrees],
        textposition='middle center',
        textfont=dict(color='white', size=10),
        hovertext=[f"{labels[i]}<br>Degree: {degrees[i]}" for i in range(len(vertices))],
        hoverinfo='text',
        showlegend=False
    ))
    
    fig.update_layout(
        title='Network Colored by Node Degree (Connectivity)',
        xaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
        yaxis=dict(showgrid=False, zeroline=False, showticklabels=False, scaleanchor='x'),
        plot_bgcolor='#1a1a2e',
        paper_bgcolor='#1a1a2e',
        font=dict(color='white'),
        width=800,
        height=600
    )
    
    return fig

fig_degree = visualize_by_degree(graph, room_names)
fig_degree.show()

## Connectivity Analysis

In [ ]:
# Analyze connectivity
vertices = graph.Vertices()
degrees = [graph.VertexDegree(v) for v in vertices]

print("Room Connectivity Analysis")
print("=" * 50)

# Sort by degree
sorted_rooms = sorted(zip(room_names, room_types, degrees), key=lambda x: -x[2])

for name, rtype, degree in sorted_rooms:
    print(f"  {name:20s} [{rtype:20s}] - {degree} connections")

print(f"\nStatistics:")
print(f"  Most connected: {sorted_rooms[0][0]} ({sorted_rooms[0][2]} connections)")
print(f"  Least connected: {sorted_rooms[-1][0]} ({sorted_rooms[-1][2]} connections)")
print(f"  Average connections: {sum(degrees)/len(degrees):.2f}")

## Note on Pyvis Export

The original topologicpy notebook uses `Graph.PyvisGraph()` to export interactive HTML network visualizations using the Pyvis library. This feature is not currently available in topologic_fast.

**Features not available in topologic_fast:**
- `Graph.PyvisGraph()` - Export to Pyvis HTML visualization
- Dictionary-based vertex/edge styling with keys like 'vertexLabelKey', 'vertexGroupKey'
- Automatic color scaling by vertex groups

However, you can achieve similar interactive visualizations using:
1. **Plotly** - As demonstrated in this notebook (works in Jupyter)
2. **NetworkX + Pyvis** - Export graph data to NetworkX, then use Pyvis directly

In [ ]:
# Example: Export to Pyvis using NetworkX (if available)
try:
    import networkx as nx
    from pyvis.network import Network
    
    # Create NetworkX graph
    G = nx.Graph()
    
    vertices = graph.Vertices()
    degrees = [graph.VertexDegree(v) for v in vertices]
    
    # Add nodes
    for i, v in enumerate(vertices):
        coords = v.Coordinates()
        label = room_names[i] if i < len(room_names) else f"Node {i}"
        rtype = room_types[i] if i < len(room_types) else "unknown"
        color = color_map.get(rtype, '#808080')
        size = 10 + degrees[i] * 5
        
        G.add_node(i, 
                   label=label, 
                   title=f"{label}\nType: {rtype}\nDegree: {degrees[i]}",
                   color=color,
                   size=size,
                   x=coords[0] * 50,
                   y=coords[1] * 50)
    
    # Add edges
    for i, v in enumerate(vertices):
        adjacent = graph.AdjacentVertices(v)
        for adj_v in adjacent:
            adj_coords = adj_v.Coordinates()
            for j, v2 in enumerate(vertices):
                v2_coords = v2.Coordinates()
                if (abs(adj_coords[0] - v2_coords[0]) < 0.01 and
                    abs(adj_coords[1] - v2_coords[1]) < 0.01 and
                    abs(adj_coords[2] - v2_coords[2]) < 0.01):
                    if i < j:  # Avoid duplicates
                        G.add_edge(i, j)
                    break
    
    # Create Pyvis network
    net = Network(notebook=True, height="600px", width="100%", bgcolor="#222222", font_color="white")
    net.from_nx(G)
    net.toggle_physics(True)
    
    # Save to HTML
    output_path = "pyvis_graph.html"
    net.show(output_path)
    print(f"Pyvis graph saved to: {output_path}")
    
except ImportError as e:
    print(f"Pyvis/NetworkX not available: {e}")
    print("Install with: pip install pyvis networkx")
    print("\nUsing Plotly for visualization instead (shown above).")

## Combined View: Geometry + Network

In [ ]:
def visualize_building_with_network(cellcomplex, graph, room_colors, room_names):
    """
    Show building geometry with network overlay.
    """
    fig = go.Figure()
    
    cells = cellcomplex.Cells()
    
    # Draw cells with transparency
    for i, cell in enumerate(cells):
        faces = cell.Faces()
        color = room_colors[i] if i < len(room_colors) else '#808080'
        
        for j, face in enumerate(faces):
            vertices = face.Vertices()
            coords = [v.Coordinates() for v in vertices]
            
            if len(coords) >= 3:
                x = [c[0] for c in coords]
                y = [c[1] for c in coords]
                z = [c[2] for c in coords]
                
                fig.add_trace(go.Mesh3d(
                    x=x, y=y, z=z,
                    color=color,
                    opacity=0.2,
                    alphahull=0,
                    name=room_names[i] if i < len(room_names) else f"Cell {i}",
                    showlegend=(j == 0)
                ))
    
    # Draw graph edges
    graph_edges = graph.Edges()
    for edge in graph_edges:
        edge_verts = edge.Vertices()
        if len(edge_verts) == 2:
            p1 = edge_verts[0].Coordinates()
            p2 = edge_verts[1].Coordinates()
            fig.add_trace(go.Scatter3d(
                x=[p1[0], p2[0]],
                y=[p1[1], p2[1]],
                z=[p1[2], p2[2]],
                mode='lines',
                line=dict(color='cyan', width=6),
                showlegend=False,
                hoverinfo='skip'
            ))
    
    # Draw graph vertices
    graph_vertices = graph.Vertices()
    vertex_coords = [v.Coordinates() for v in graph_vertices]
    x = [c[0] for c in vertex_coords]
    y = [c[1] for c in vertex_coords]
    z = [c[2] for c in vertex_coords]
    degrees = [graph.VertexDegree(v) for v in graph_vertices]
    
    fig.add_trace(go.Scatter3d(
        x=x, y=y, z=z,
        mode='markers',
        marker=dict(
            size=[6 + d * 2 for d in degrees],
            color='cyan',
            line=dict(color='white', width=1)
        ),
        name='Network Nodes',
        hovertext=[f"{room_names[i]}<br>Degree: {degrees[i]}" if i < len(room_names) else f"Node {i}" for i in range(len(graph_vertices))],
        hoverinfo='text'
    ))
    
    fig.update_layout(
        title='Building Geometry with Connectivity Network',
        scene=dict(
            aspectmode='data',
            xaxis_title='X',
            yaxis_title='Y',
            zaxis_title='Z',
            camera=dict(eye=dict(x=1.5, y=-1.5, z=1.0))
        ),
        width=900,
        height=700
    )
    
    return fig

fig_combined = visualize_building_with_network(building, graph, room_colors, room_names)
fig_combined.show()

## Summary

This notebook demonstrated interactive network visualization using topologic_fast:

1. **Creating Complex Geometry** - Multi-room building with CellComplex
2. **Dual Graph Extraction** - Room connectivity via `Graph.ByTopology()`
3. **Interactive 2D Networks** - Using Plotly Scatter plots
4. **3D Network Visualization** - Using Plotly Scatter3d
5. **Node Grouping by Type** - Color-coded by room function
6. **Degree-based Visualization** - Size/color by connectivity

### Key topologic_fast Methods Used:
- `tf.Cell.Box()` - Create box cells
- `tf.CellComplex.ByCells()` - Combine cells
- `tf.Graph.ByTopology()` - Extract dual graph
- `graph.Vertices()`, `graph.Edges()` - Access graph elements
- `graph.VertexDegree()` - Get node connectivity
- `graph.AdjacentVertices()` - Get neighboring nodes